In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import numpy as np
import seaborn as sns
import seaborn as sns
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
import optuna
from sklearn.ensemble import RandomForestRegressor
import json
from catboost import CatBoostRegressor


c:\Users\Mohsen\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Load data

# data = pd.read_csv("D:/Drive E/Papers/Diabet/140308/Shapley/app/normal.csv")
# data.columns = ['C', 'FAg', 'CA', 'W', 'WRA', 'FA', 'ACC', 'SF', 'Age', 'CS']
data = pd.read_csv("D:/Drive E/Papers/Diabet/140308/Shapley/app/hpc.csv")
# data.columns = ['C', 'BFS', 'FA', 'W', 'SP', 'CA', 'FAg', 'Age', 'CS']
# data = pd.read_csv("D:/Drive E/Papers/Diabet/140308/Shapley/app/hsc.csv")
# datolumns = ['C', 'SF', 'W', 'SP', 'FA', 'CA', 'Age', 'CS']
#data = pd.read_csv("D:/Drive E/Papers/Diabet/140308/Shapley/app/UHPC.csv")
#data.columns = ['C', 'BFS', 'SF', 'LP', 'QP', 'FA', 'NS', 'W', 'FAg', 'CA', 'Fi', 'SP', 'RH', 'T', 'Age', 'CS']

X = data.iloc[:, :-1]
y = data.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
#data.describe()

In [ ]:
# Correlation map of concrete mix variables and cost. 

corr = data.corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, square=True, linewidths=0.5)
plt.title("Correlation Map of Concrete Mix Variables", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Tuning the hyperparameters of the AdaBoostRegressor model

def objective(trial):

    base_estimator = DecisionTreeRegressor(
        max_depth=trial.suggest_int("max_depth", 1, 10),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
        random_state=42
    )

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 1.0, log=True),
        "loss": trial.suggest_categorical("loss", ["linear", "square", "exponential"]),
        "estimator": base_estimator,
        "random_state": 42
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse_list = []

    for tr_idx, val_idx in kf.split(X_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        model = AdaBoostRegressor(**params)
        model.fit(X_tr, y_tr)

        y_pred = model.predict(X_val)
        rmse_list.append(np.sqrt(mean_squared_error(y_val, y_pred)))

    return np.mean(rmse_list)
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)
for key, value in study.best_params.items():
    print(f"{key}: {value}")

In [6]:
# Evaluating the AdaBoostRegressor model
final_model = AdaBoostRegressor(
    n_estimators=546,
    learning_rate= 0.03658499625081239,
    loss="exponential",
    random_state=42,
    estimator=DecisionTreeRegressor(
        max_depth=9,
        min_samples_split=5,
        min_samples_leaf=1,
        random_state=42
    )
)

# normal
final_model = AdaBoostRegressor(
    n_estimators=134,
    learning_rate= 0.023970995,
    loss="linear",
    random_state=42,
    estimator=DecisionTreeRegressor(
        max_depth=9,
        min_samples_split=8,
        min_samples_leaf=2,
        random_state=42
    )
)
# hpc
final_model = AdaBoostRegressor(
    n_estimators=461,
    learning_rate= 0.011936645,
    loss="square",
    random_state=42,
    estimator=DecisionTreeRegressor(
        max_depth=9,
        min_samples_split=9,
        min_samples_leaf=1,
        random_state=42
    )
)
# #hsc
# final_model = AdaBoostRegressor(
#     n_estimators=546,
#     learning_rate= 0.036584996,
#     loss="exponential",
#     random_state=42,
#     estimator=DecisionTreeRegressor(
#         max_depth=9,
#         min_samples_split=5,
#         min_samples_leaf=1,
#         random_state=42
#     )
# )
# #uhpc
# final_model = AdaBoostRegressor(
#     n_estimators=765,
#     learning_rate= 0.077007869,
#     loss="square",
#     random_state=42,
#     estimator=DecisionTreeRegressor(
#         max_depth=10,
#         min_samples_split=13,
#         min_samples_leaf=5,
#         random_state=42
#     )
# )
# # RAC
# final_model = AdaBoostRegressor(
#     n_estimators=903,
#     learning_rate= 0.019405336,
#     loss="linear",
#     random_state=42,
#     estimator=DecisionTreeRegressor(
#         max_depth=8,
#         min_samples_split=6,
#         min_samples_leaf=2,
#         random_state=42
#     )
# )
final_model.fit(X_train, y_train)
models = {}
models['AdaBoostRegressor'] = final_model  
y_pred = final_model.predict(X_test)
y_AdaBoostRegressor=y_pred
model_AdaBoostRegressor=final_model


r2 = r2_score(y_test, y_pred) #Coefficient of determination
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
epsilon = 1e-8   
mean_absolute_relative_error = np.mean(np.abs((y_test - y_pred) / (y_test + epsilon)))
mean_square_relative_error=np.mean(((y_test - y_pred) / (y_test + epsilon))**2)
Root_mean_squared_relative_error=np.sqrt(mean_square_relative_error)
Relative_root_mean_square_error=rmse/np.sum(y_test)*100
mean_bias_error=np.mean(y_test - y_pred)
max_absolute_relative_error=np.max(np.abs((y_test - y_pred) / (y_test + epsilon)))
std_diff = np.std(y_test - y_pred, ddof=0)
uncertainty_95=np.sqrt(1.96 * (std_diff**2 + rmse**2))
r2, np.corrcoef(y_test, y_pred)[0,1], mse, rmse, mae

(0.9145233247144093,
 0.958183948574123,
 34.277977055464945,
 5.8547397086006265,
 4.257350074277063)

In [ ]:
# Tuning the hyperparameters of the RandomForestRegressor

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1500),
        "max_depth": trial.suggest_int("max_depth", 3, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        "random_state": 42,
        "n_jobs": -1
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse_list = []

    for tr_idx, val_idx in kf.split(X_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        model = RandomForestRegressor(**params)
        model.fit(X_tr, y_tr)

        y_pred = model.predict(X_val)
        rmse_list.append(np.sqrt(mean_squared_error(y_val, y_pred)))

    return np.mean(rmse_list)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)
for key, value in study.best_params.items():
    print(f"{key}: {value}")


In [7]:
# Evaluating the RandomForestRegressor model
final_model = RandomForestRegressor(
    n_estimators=555,
    max_depth=26,
    min_samples_split=4,
    min_samples_leaf=1,
    max_features=None,
    bootstrap=False,
    random_state=42,   
)
#normal
final_model = RandomForestRegressor(
    n_estimators=1094,
    max_depth=19,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    bootstrap=True,
    random_state=42,  
)
# hpc
final_model = RandomForestRegressor(
    n_estimators=467,
    max_depth=30,
    min_samples_split=6,
    min_samples_leaf=1,
    max_features='log2',
    bootstrap=False,
    random_state=42,    
)
# #hsc
# final_model = RandomForestRegressor(
#     n_estimators=555,
#     max_depth=26,
#     min_samples_split=4,
#     min_samples_leaf=1,
#     max_features=None,
#     bootstrap=False,
#     random_state=42,    
# )
# #uhpc
# final_model = RandomForestRegressor(
#     n_estimators=464,
#     max_depth=10,
#     min_samples_split=4,
#     min_samples_leaf=2,
#     max_features=None,
#     bootstrap=True,
#     random_state=42,    
# )
# #RAC
# final_model = RandomForestRegressor(
#     n_estimators=1482,
#     max_depth=11,
#     min_samples_split=2,
#     min_samples_leaf=1,
#     max_features='log2',
#     bootstrap=False,
#     random_state=42,    
# )
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)
y_RandomForestRegressor=y_pred
model_RandomForestRegressor=final_model

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)
mare = np.mean(np.abs((y_test - y_pred) / y_test))

r2 = r2_score(y_test, y_pred) #Coefficient of determination
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
epsilon = 1e-8   
mean_absolute_relative_error = np.mean(np.abs((y_test - y_pred) / (y_test + epsilon)))
mean_square_relative_error=np.mean(((y_test - y_pred) / (y_test + epsilon))**2)
Root_mean_squared_relative_error=np.sqrt(mean_square_relative_error)
Relative_root_mean_square_error=rmse/np.sum(y_test)
mean_bias_error=np.mean(y_test - y_pred)
max_absolute_relative_error=np.max(np.abs((y_test - y_pred) / (y_test + epsilon)))
std_diff = np.std(y_test - y_pred, ddof=0)
uncertainty_95=np.sqrt(1.96 * (std_diff**2 + rmse**2))
r2, np.corrcoef(y_test, y_pred)[0,1], mse, rmse, mae

(0.9358867108357265,
 0.9708485326920846,
 25.710801778152845,
 5.070581995999359,
 3.513508697418184)

In [ ]:
# Tuning the hyperparameters of the SVR

from sklearn.svm import SVR
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import numpy as np

def objective(trial):
    params = {
        "C": trial.suggest_float("C", 0.1, 100, log=True),
        "epsilon": trial.suggest_float("epsilon", 0.001, 1.0, log=True),
        "gamma": trial.suggest_float("gamma", 1e-4, 1e-1, log=True),
        "kernel": trial.suggest_categorical("kernel", ["rbf", "poly", "sigmoid"])
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse_list = []

    for tr_idx, val_idx in kf.split(X_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("svr", SVR(**params))
        ])

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)
        y_SVR=y_pred

        rmse_list.append(np.sqrt(mean_squared_error(y_val, y_pred)))

    return np.mean(rmse_list)
import optuna
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)
for key, value in study.best_params.items():
    print(f"{key}: {value}")


In [ ]:
# Evaluating the SVR model

from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error,mean_squared_error

final_model = Pipeline([
            ("scaler", StandardScaler()),
            ("svr", SVR(C=56.419471276084636, 
                        epsilon=0.010881127269250368,
                        gamma=0.09279425119531859,
                        kernel='rbf'))
        ])
# normal
final_model = Pipeline([
            ("scaler", StandardScaler()),
            ("svr", SVR(C=51.7632911, 
                        epsilon=0.401105595,
                        gamma=0.069708146,
                        kernel='rbf'))
        ])
# hpc
final_model = Pipeline([
            ("scaler", StandardScaler()),
            ("svr", SVR(C=30.61330396, 
                        epsilon=0.013310678,
                        gamma=0.090972353,
                        kernel='rbf'))
        ])
#hsc
final_model = Pipeline([
            ("scaler", StandardScaler()),
            ("svr", SVR(C=56.41947128, 
                        epsilon=0.010881127,
                        gamma=0.092794251,
                        kernel='rbf'))
        ])
#uhpc
final_model = Pipeline([
            ("scaler", StandardScaler()),
            ("svr", SVR(C=46.61179444, 
                        epsilon=0.003224848,
                        gamma=0.006240595,
                        kernel='sigmoid'))
        ])
#RAC
final_model = Pipeline([
            ("scaler", StandardScaler()),
            ("svr", SVR(C=99.36496978, 
                        epsilon=0.003879942,
                        gamma=0.049123813,
                        kernel='rbf'))
        ])
final_model.fit(X_train, y_train)
model_SVR=final_model
y_pred = final_model.predict(X_test)
y_SVR = y_pred
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
mare = np.mean(np.abs((y_test - y_pred) / y_test))
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

r2 = r2_score(y_test, y_pred) #Coefficient of determination
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
epsilon = 1e-8   
mean_absolute_relative_error = np.mean(np.abs((y_test - y_pred) / (y_test + epsilon)))
mean_square_relative_error=np.mean(((y_test - y_pred) / (y_test + epsilon))**2)
Root_mean_squared_relative_error=np.sqrt(mean_square_relative_error)
Relative_root_mean_square_error=rmse/np.sum(y_test)
mean_bias_error=np.mean(y_test - y_pred)
max_absolute_relative_error=np.max(np.abs((y_test - y_pred) / (y_test + epsilon)))
std_diff = np.std(y_test - y_pred, ddof=0)
uncertainty_95=np.sqrt(1.96 * (std_diff**2 + rmse**2))
r2, np.corrcoef(y_test, y_pred)[0,1], mse, rmse, mae

In [ ]:
# Tuning the hyperparameters of the XGBRegressor

from xgboost import XGBRegressor
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1500),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "subsample": trial.suggest_float("subsample", 0.3, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "random_state": 42,
        "n_jobs": -1
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse_list = []

    for tr_idx, val_idx in kf.split(X_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        model = XGBRegressor(**params)
        model.fit(X_tr, y_tr)

        y_pred = model.predict(X_val)
        rmse_list.append(np.sqrt(mean_squared_error(y_val, y_pred)))

    return np.mean(rmse_list)
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)
for key, value in study.best_params.items():
    print(f"{key}: {value}")


In [4]:
# Evaluating the XGBRegressor model
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error
final_model = XGBRegressor(
    n_estimators=973,
    max_depth=4,
    gamma= 0.17420784613936036,
    colsample_bytree= 0.726895124558764,
    subsample= 0.4123131099152587,
    min_child_weight=3,
    learning_rate= 0.19074296073264566,
    random_state=42,
    n_jobs=-1
)
#normal
final_model = XGBRegressor(
    n_estimators=1346,
    max_depth=4,
    gamma= 1.262401114,
    colsample_bytree= 0.895940613,
    subsample= 0.990397524,
    min_child_weight=5,
    learning_rate= 0.046783977,
    random_state=42,
    n_jobs=-1
)
#hpc
final_model = XGBRegressor(
    n_estimators=1408,
    max_depth=10,
    gamma= 0.496671009,
    colsample_bytree= 0.853814347,
    subsample= 0.404053905,
    min_child_weight=8,
    learning_rate= 0.032692385,
    random_state=42,
    n_jobs=-1
)
# #hsc
# final_model = XGBRegressor(
#     n_estimators=973,
#     max_depth=4,
#     gamma= 0.174207846,
#     colsample_bytree= 0.726895125,
#     subsample= 0.41231311,
#     min_child_weight=3,
#     learning_rate= 0.190742961,
#     random_state=42,
#     n_jobs=-1
# )
# #uhpc
# final_model = XGBRegressor(
#     n_estimators=807,
#     max_depth=12,
#     gamma= 1.624678716,
#     colsample_bytree= 0.574570345,
#     subsample= 0.567406563,
#     min_child_weight=9,
#     learning_rate= 0.118934836,
#     random_state=42,
#     n_jobs=-1
# )
# #rac
# final_model = XGBRegressor(
#     n_estimators=673,
#     max_depth=8,
#     gamma= 4.471614342,
#     colsample_bytree= 0.932883273,
#     subsample= 0.411045476,
#     min_child_weight=4,
#     learning_rate= 0.140932069,
#     random_state=42,
#     n_jobs=-1
# )
final_model.fit(X_train, y_train)
model_XGBRegressor=final_model
y_pred = final_model.predict(X_test)
y_XGBRegressor=y_pred
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
from sklearn.metrics import mean_squared_error, r2_score

r2 = r2_score(y_test, y_pred)
r2 = r2_score(y_test, y_pred) #Coefficient of determination
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
epsilon = 1e-8   
mean_absolute_relative_error = np.mean(np.abs((y_test - y_pred) / (y_test + epsilon)))
mean_square_relative_error=np.mean(((y_test - y_pred) / (y_test + epsilon))**2)
Root_mean_squared_relative_error=np.sqrt(mean_square_relative_error)
Relative_root_mean_square_error=rmse/np.sum(y_test)
mean_bias_error=np.mean(y_test - y_pred)
max_absolute_relative_error=np.max(np.abs((y_test - y_pred) / (y_test + epsilon)))
std_diff = np.std(y_test - y_pred, ddof=0)
uncertainty_95=np.sqrt(1.96 * (std_diff**2 + rmse**2))
r2, np.corrcoef(y_test, y_pred)[0,1], mse, rmse, mae

(0.969240714442291,
 0.9850428491686102,
 12.3351321406319,
 3.5121406777963635,
 2.296532965022556)

In [ ]:
# Tuning the hyperparameters of the MLPRegressor

from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import numpy as np

def objective(trial):
    n_layers = trial.suggest_int("n_layers", 1, 3)
    hidden_sizes = []

    for i in range(n_layers):
        hidden_sizes.append(trial.suggest_int(f"n_units_layer_{i}", 16, 256))

    hidden_layer_sizes = tuple(hidden_sizes)

    params = {
        "hidden_layer_sizes": hidden_layer_sizes,
        "activation": trial.suggest_categorical("activation", ["relu", "tanh", "logistic"]),
        "solver": trial.suggest_categorical("solver", ["adam", "lbfgs"]),
        "alpha": trial.suggest_float("alpha", 1e-6, 1e-1, log=True),
        "learning_rate_init": trial.suggest_float("learning_rate_init", 1e-4, 1e-1, log=True),
        "max_iter": 1000,
        "random_state": 42
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse_list = []

    for tr_idx, val_idx in kf.split(X_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("ann", MLPRegressor(**params))
        ])

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)
        rmse_list.append(np.sqrt(mean_squared_error(y_val, y_pred)))

    return np.mean(rmse_list)
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)
for key, value in study.best_params.items():
    print(f"{key}: {value}")


In [ ]:
# Evaluating the MLPRegressor model
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import pandas as pd
final_model = MLPRegressor(
    hidden_layer_sizes= (169),
    activation= 'relu',
    solver= 'adam',
    alpha= 0.0014792113473951134,
    learning_rate_init= 0.033233780399357746,
    max_iter= 1000,
    random_state= 42
)
#normal
final_model = MLPRegressor(
    hidden_layer_sizes= (41),
    activation= 'logistic',
    solver= 'adam',
    alpha= 0.000371636,
    learning_rate_init= 0.078488861,
    max_iter= 1000,
    random_state= 42
)
#hpc
final_model = MLPRegressor(
    hidden_layer_sizes= (187,71),
    activation= 'tanh',
    solver= 'adam',
    alpha= 0.000218255,
    learning_rate_init= 0.001681669,
    max_iter= 1000,
    random_state= 42
)
#hsc
final_model = MLPRegressor(
    hidden_layer_sizes= (169),
    activation= 'relu',
    solver= 'adam',
    alpha= 0.001479211,
    learning_rate_init= 0.03323378,
    max_iter= 1000,
    random_state= 42
)
#uhpc
final_model = MLPRegressor(
    hidden_layer_sizes= (81,186),
    activation= 'tanh',
    solver= 'lbfgs',
    alpha= 0.027318343,
    learning_rate_init= 0.000952438,
    max_iter= 1000,
    random_state= 42
)
#rac
final_model = MLPRegressor(
    hidden_layer_sizes= (212,89),
    activation= 'relu',
    solver= 'adam',
    alpha= 0.059967089,
    learning_rate_init= 0.002842192,
    max_iter= 1000,
    random_state= 42
)
final_model = Pipeline([
            ("scaler", StandardScaler()),
            ("ann", MLPRegressor(hidden_layer_sizes= (41), activation= 'logistic', solver= 'adam', alpha= 0.00037163632373739226, learning_rate_init= 0.07848886079973288, max_iter= 2000, random_state= 42))
        ])
# final_model = Pipeline([
#             ("scaler", MinMaxScaler(feature_range=(-1, 1))),
#             ("ann", MLPRegressor(hidden_layer_sizes= (41), activation= 'logistic', solver= 'adam', alpha= 0.00037163632373739226, learning_rate_init= 0.07848886079973288, max_iter= 2000, random_state= 42))
#         ])
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)
y_MLPRegressor=y_pred
model_MLPRegressor=final_model


mare = np.mean(np.abs((y_test - y_pred) / y_test))
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

r2 = r2_score(y_test, y_pred) #Coefficient of determination
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
epsilon = 1e-8  
mean_absolute_relative_error = np.mean(np.abs((y_test - y_pred) / (y_test + epsilon)))
mean_square_relative_error=np.mean(((y_test - y_pred) / (y_test + epsilon))**2)
Root_mean_squared_relative_error=np.sqrt(mean_square_relative_error)
Relative_root_mean_square_error=rmse/np.sum(y_test)
mean_bias_error=np.mean(y_test - y_pred)
max_absolute_relative_error=np.max(np.abs((y_test - y_pred) / (y_test + epsilon)))
std_diff = np.std(y_test - y_pred, ddof=0)
uncertainty_95=np.sqrt(1.96 * (std_diff**2 + rmse**2))
r2, np.corrcoef(y_test, y_pred)[0,1], mse, rmse, mae

In [ ]:
# Tuning the hyperparameters of the CatBoostRegressor

from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import numpy as np

def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 200, 1500),
        "depth": trial.suggest_int("depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 5),
        "random_strength": trial.suggest_float("random_strength", 0, 2),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "loss_function": "RMSE",
        "random_seed": 42,
        "verbose": 0
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse_list = []

    for tr_idx, val_idx in kf.split(X_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        model = CatBoostRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=0)

        y_pred = model.predict(X_val)
        rmse_list.append(np.sqrt(mean_squared_error(y_val, y_pred)))

    return np.mean(rmse_list)
import optuna
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)
for key, value in study.best_params.items():
    print(f"{key}: {value}")


In [ ]:
# Evaluating the CatBoostRegressor model

from catboost import CatBoostRegressor
final_model = CatBoostRegressor(
    iterations= 1382,
    depth= 3,
    learning_rate= 0.21002345470863443,
    l2_leaf_reg= 4.564095858449578,
    bagging_temperature= 1.4741942333299374,
    random_strength= 1.5087103329667955,
    border_count= 245,
    random_state=42
)
#normal
final_model = CatBoostRegressor(
    iterations= 1271,
    depth= 4,
    learning_rate= 0.224459481,
    l2_leaf_reg= 4.564095858449578,
    bagging_temperature= 1.021483044,
    random_strength= 4.010189783,
    border_count= 208,
    random_state=42
)
#hpc
final_model = CatBoostRegressor(
    iterations= 1150,
    depth= 4,
    learning_rate= 0.114777392,
    l2_leaf_reg= 6.942658041,
    bagging_temperature= 3.673767272,
    random_strength= 1.529709068,
    border_count= 38,
    random_state=42
)
#hsc
final_model = CatBoostRegressor(
    iterations= 1382,
    depth= 3,
    learning_rate= 0.210023455,
    l2_leaf_reg= 4.564095858,
    bagging_temperature= 1.474194233,
    random_strength= 1.508710333,
    border_count= 245,
    random_state=42
)
#uhpc
final_model = CatBoostRegressor(
    iterations= 1462,
    depth= 7,
    learning_rate= 0.082467944,
    l2_leaf_reg= 4.401110894,
    bagging_temperature= 0.082833561,
    random_strength= 1.609339634,
    border_count= 62,
    random_state=42
)
#rac
1421
5
0.234333852
9.540618068
1.735015718
1.00651784
238
final_model = CatBoostRegressor(
    iterations= 1421,
    depth= 5,
    learning_rate= 0.234333852,
    l2_leaf_reg= 9.540618068,
    bagging_temperature= 1.735015718,
    random_strength= 1.00651784,
    border_count= 238,
    random_state=42
)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)
y_CatBoostRegressor=y_pred
model_CatBoostRegressor = final_model
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

r2 = r2_score(y_test, y_pred)
r2 = r2_score(y_test, y_pred) #Coefficient of determination
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
epsilon = 1e-8   
mean_absolute_relative_error = np.mean(np.abs((y_test - y_pred) / (y_test + epsilon)))
mean_square_relative_error=np.mean(((y_test - y_pred) / (y_test + epsilon))**2)
Root_mean_squared_relative_error=np.sqrt(mean_square_relative_error)
Relative_root_mean_square_error=rmse/np.sum(y_test)
mean_bias_error=np.mean(y_test - y_pred)
max_absolute_relative_error=np.max(np.abs((y_test - y_pred) / (y_test + epsilon)))
std_diff = np.std(y_test - y_pred, ddof=0)
uncertainty_95=np.sqrt(1.96 * (std_diff**2 + rmse**2))
r2, np.corrcoef(y_test, y_pred)[0,1], mse, rmse, mae

In [94]:
models = {
    "XGBRegressor": model_XGBRegressor,
    "MLPRegressor": model_MLPRegressor,
    "SVR": model_SVR,
    "AdaBoostRegressor": model_AdaBoostRegressor,
    "RandomForestRegressor": model_RandomForestRegressor,
    "CatBoostRegressor": model_CatBoostRegressor
}

In [ ]:
# Comparison of R² scores models across 10-fold cross-validation.

import matplotlib.pyplot as plt
kf = KFold(n_splits=5, shuffle=True, random_state=42)

model_names = list(models.keys())
fold_r2 = {name: [] for name in model_names}


for model_name, model in models.items():
    for fold, (train_idx, test_idx) in enumerate(kf.split(X, y), start=1):
        X_train1, X_test1 = X.iloc[train_idx], X.iloc[test_idx]
        y_train1, y_test1 = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(X_train1, y_train1)
        y_pred1 = model.predict(X_test1)
        r2 = r2_score(y_test1, y_pred1)

        fold_r2[model_name].append(r2)



plt.figure(figsize=(18, 7))

n_folds = 10
n_models = len(models)
x = np.arange(n_folds)  
width = 0.15            

colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple", "tab:brown"]

for i, model_name in enumerate(model_names):
    plt.bar(x + i*width, fold_r2[model_name], width, label=model_name, color=colors[i])

plt.xlabel("Fold Number", fontsize=12)
plt.ylabel("R² Score", fontsize=12)
plt.title("Comparison of 6 Models Across 10 Folds (R² Score)", fontsize=14)

plt.xticks(x + width*2, [f"Fold {i}" for i in range(1, 11)])
plt.legend(title="Models")
plt.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# Comparison of actual and predicted values for both training and testing datasets.

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score

final_model = model_AdaBoostRegressor
y_pred = final_model.predict(X_test)

r2_test = r2_score(y_test, y_AdaBoostRegressor)
r2_train = r2_score(y_train, final_model.predict(X_train))

plt.scatter(y_test, y_pred, label='test')
plt.scatter(y_train, final_model.predict(X_train), label='train')

min_val = min(min(y_test), min(y_train))
max_val = max(max(y_test), max(y_train))
x_vals = np.linspace(min_val, max_val, 100)

plt.plot(x_vals, x_vals, 'k--', label='y = x')

plt.plot(x_vals, x_vals * 1.10, 'r--', label='+10% deviation')
plt.plot(x_vals, x_vals * 0.90, 'b--', label='-10% deviation')

textstr = f'R² test = {r2_test:.3f}\nR² train = {r2_train:.3f}'
plt.text(0.95, 0.05, textstr, transform=plt.gca().transAxes,
         fontsize=10, verticalalignment='bottom', horizontalalignment='right',
         bbox=dict(facecolor='white', alpha=0.6, edgecolor='gray'))

plt.legend()
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('AdaBoost')
plt.show()

In [ ]:
# Comparison of actual and predicted values for both training and testing datasets.

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score

final_model = model_RandomForestRegressor
y_pred = final_model.predict(X_test)

r2_test = r2_score(y_test, y_RandomForestRegressor)
r2_train = r2_score(y_train, final_model.predict(X_train))

plt.scatter(y_test, y_pred, label='test')
plt.scatter(y_train, final_model.predict(X_train), label='train')

min_val = min(min(y_test), min(y_train))
max_val = max(max(y_test), max(y_train))
x_vals = np.linspace(min_val, max_val, 100)

plt.plot(x_vals, x_vals, 'k--', label='y = x')

plt.plot(x_vals, x_vals * 1.10, 'r--', label='+10% deviation')
plt.plot(x_vals, x_vals * 0.90, 'b--', label='-10% deviation')

textstr = f'R² test = {r2_test:.3f}\nR² train = {r2_train:.3f}'
plt.text(0.95, 0.05, textstr, transform=plt.gca().transAxes,
         fontsize=10, verticalalignment='bottom', horizontalalignment='right',
         bbox=dict(facecolor='white', alpha=0.6, edgecolor='gray'))

plt.legend()
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('RF')
plt.show()

In [ ]:
# Comparison of actual and predicted values for both training and testing datasets.

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score

final_model = model_SVR
y_pred = final_model.predict(X_test)

r2_test = r2_score(y_test, y_SVR)
r2_train = r2_score(y_train, final_model.predict(X_train))

plt.scatter(y_test, y_pred, label='test')
plt.scatter(y_train, final_model.predict(X_train), label='train')

min_val = min(min(y_test), min(y_train))
max_val = max(max(y_test), max(y_train))
x_vals = np.linspace(min_val, max_val, 100)

plt.plot(x_vals, x_vals, 'k--', label='y = x')

plt.plot(x_vals, x_vals * 1.10, 'r--', label='+10% deviation')
plt.plot(x_vals, x_vals * 0.90, 'b--', label='-10% deviation')

textstr = f'R² test = {r2_test:.3f}\nR² train = {r2_train:.3f}'
plt.text(0.95, 0.05, textstr, transform=plt.gca().transAxes,
         fontsize=10, verticalalignment='bottom', horizontalalignment='right',
         bbox=dict(facecolor='white', alpha=0.6, edgecolor='gray'))

plt.legend()
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('SVR')
plt.show()

In [ ]:
# Comparison of actual and predicted values for both training and testing datasets.

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score

final_model = model_XGBRegressor
y_pred = final_model.predict(X_test)

r2_test = r2_score(y_test, y_XGBRegressor)
r2_train = r2_score(y_train, final_model.predict(X_train))

plt.scatter(y_test, y_pred, label='test')
plt.scatter(y_train, final_model.predict(X_train), label='train')

min_val = min(min(y_test), min(y_train))
max_val = max(max(y_test), max(y_train))
x_vals = np.linspace(min_val, max_val, 100)

plt.plot(x_vals, x_vals, 'k--', label='y = x')

plt.plot(x_vals, x_vals * 1.10, 'r--', label='+10% deviation')
plt.plot(x_vals, x_vals * 0.90, 'b--', label='-10% deviation')

textstr = f'R² test = {r2_test:.3f}\nR² train = {r2_train:.3f}'
plt.text(0.95, 0.05, textstr, transform=plt.gca().transAxes,
         fontsize=10, verticalalignment='bottom', horizontalalignment='right',
         bbox=dict(facecolor='white', alpha=0.6, edgecolor='gray'))

plt.legend()
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('XGBoost')
plt.show()

In [ ]:
# Comparison of actual and predicted values for both training and testing datasets.

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score

final_model = model_MLPRegressor
y_pred = final_model.predict(X_test)

r2_test = r2_score(y_test, y_MLPRegressor)
r2_train = r2_score(y_train, final_model.predict(X_train))

plt.scatter(y_test, y_pred, label='test')
plt.scatter(y_train, final_model.predict(X_train), label='train')

min_val = min(min(y_test), min(y_train))
max_val = max(max(y_test), max(y_train))
x_vals = np.linspace(min_val, max_val, 100)

plt.plot(x_vals, x_vals, 'k--', label='y = x')

plt.plot(x_vals, x_vals * 1.10, 'r--', label='+10% deviation')
plt.plot(x_vals, x_vals * 0.90, 'b--', label='-10% deviation')

textstr = f'R² test = {r2_test:.3f}\nR² train = {r2_train:.3f}'
plt.text(0.95, 0.05, textstr, transform=plt.gca().transAxes,
         fontsize=10, verticalalignment='bottom', horizontalalignment='right',
         bbox=dict(facecolor='white', alpha=0.6, edgecolor='gray'))

plt.legend()
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('ANN')
plt.show()

In [ ]:
# Comparison of actual and predicted values for both training and testing datasets.

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score

final_model = model_CatBoostRegressor
y_pred = final_model.predict(X_test)

r2_test = r2_score(y_test, y_CatBoostRegressor)
r2_train = r2_score(y_train, final_model.predict(X_train))

plt.scatter(y_test, y_pred, label='test')
plt.scatter(y_train, final_model.predict(X_train), label='train')

min_val = min(min(y_test), min(y_train))
max_val = max(max(y_test), max(y_train))
x_vals = np.linspace(min_val, max_val, 100)

plt.plot(x_vals, x_vals, 'k--', label='y = x')

plt.plot(x_vals, x_vals * 1.10, 'r--', label='+10% deviation')
plt.plot(x_vals, x_vals * 0.90, 'b--', label='-10% deviation')

textstr = f'R² test = {r2_test:.3f}\nR² train = {r2_train:.3f}'
plt.text(0.95, 0.05, textstr, transform=plt.gca().transAxes,
         fontsize=10, verticalalignment='bottom', horizontalalignment='right',
         bbox=dict(facecolor='white', alpha=0.6, edgecolor='gray'))

plt.legend()
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('CatBoost')
plt.show()

In [15]:
# Taylor diagram 

import numpy as np
import matplotlib.pyplot as plt
import matplotlib

# -------------------------
# Taylor Diagram Class
# -------------------------
class TaylorDiagram:
    def __init__(self, refstd, fig=None, rect=111, label='Reference'):
        self.refstd = refstd

        tr = np.pi / 2
        t = np.linspace(0, tr)

        if fig is None:
            fig = plt.figure(figsize=(7, 6))

        self.fig = fig
        self.ax = fig.add_subplot(rect, polar=True)

        # Set limits
        self.ax.set_xlim([0, tr])

        # Correlation labels
        rs = np.concatenate((np.linspace(0, 0.9, 10), [0.95, 1.0]))
        self.ax.set_thetagrids(np.degrees(np.arccos(rs)),
                               labels=[f"{r:.2f}" for r in rs])

        # Radius grid (standard deviation)
        self.ax.set_rgrids(np.linspace(0, refstd * 1.5, 7))

        # Plot reference point
        self.ax.plot(0, refstd, 'ko', label=label)

    def add_sample(self, stddev, corrcoef, label, marker='o', color='r'):
        theta = np.arccos(corrcoef)
        self.ax.plot(theta, stddev, marker=marker, color=color, label=label)

    def add_contours(self, levels=5, **kwargs):
        rs, ts = np.meshgrid(np.linspace(0, self.refstd * 1.5, 100),
                             np.linspace(0, np.pi/2, 100))

        corr = np.cos(ts)
        rmse = np.sqrt(self.refstd**2 + rs**2 - 2*self.refstd*rs*corr)

        contours = self.ax.contour(ts, rs, rmse, levels, **kwargs)
        return contours


In [ ]:
# Taylor diagram 

ref_std = np.std(y_test)
std_pred = np.std(y_pred)
corr = np.corrcoef(y_test, y_pred)[0, 1]

fig = plt.figure(figsize=(7, 6))
dia = TaylorDiagram(ref_std, fig=fig, label="Reference")

dia.add_sample(np.std(y_AdaBoostRegressor), np.corrcoef(y_test, y_AdaBoostRegressor)[0,1], label="AdaBoost", color="green")
dia.add_sample(np.std(y_RandomForestRegressor), np.corrcoef(y_test, y_RandomForestRegressor)[0,1], label="RandomForest", color="orange")
dia.add_sample(np.std(y_SVR), np.corrcoef(y_test, y_SVR)[0,1], label="SVR", color="blue")
dia.add_sample(np.std(y_XGBRegressor), np.corrcoef(y_test, y_XGBRegressor)[0,1], label="XGBoost", color="red")
dia.add_sample(np.std(y_MLPRegressor), np.corrcoef(y_test, y_MLPRegressor)[0,1], label="MLP", color="yellow")
dia.add_sample(np.std(y_CatBoostRegressor), np.corrcoef(y_test, y_CatBoostRegressor)[0,1], label="CatBoost", color="pink")

dia.add_contours(levels=5, colors='0.5')

plt.legend(loc='upper right')
plt.title("Taylor Diagram")
plt.show()

In [1]:
ref_std, std_pred, corr, np.std(y_AdaBoostRegressor), np.corrcoef(y_test, y_AdaBoostRegressor)[0,1], np.std(y_RandomForestRegressor), np.corrcoef(y_test, y_RandomForestRegressor)[0,1], np.std(y_SVR), np.corrcoef(y_test, y_SVR)[0,1], np.std(y_XGBRegressor), np.corrcoef(y_test, y_XGBRegressor)[0,1], np.std(y_MLPRegressor), np.corrcoef(y_test, y_MLPRegressor)[0,1], np.std(y_CatBoostRegressor), np.corrcoef(y_test, y_CatBoostRegressor)[0,1]

NameError: name 'ref_std' is not defined

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# Model data
data = {
    'AdaBoost': {
        'R2': 0.8936338904609085,
        'r': 0.9454283955070929,
        'MSE': 17.392965740306362,
        'RMSE': 4.170487470345207,
        'MAE': 2.8773415133882962
    },
    'RF': {
        'R2': 0.9010254418819429,
        'r': 0.9499213993386695,
        'MSE': 16.184300675927787,
        'RMSE': 4.02297162256059,
        'MAE': 2.7548876303562193
    },
    'SVR': {
        'R2': 0.795413370570063,
        'r': 0.893487739574029,
        'MSE': 33.453966230586616,
        'RMSE': 5.783940372322887,
        'MAE': 3.2707225028475957
    },
    'XGBoost': {
        'R2': 0.9347851857272262,
        'r': 0.9670153891913157,
        'MSE': 10.663913866191805,
        'RMSE': 3.265564861734001,
        'MAE': 2.409547672208613
    },
    'ANN': {
        'R2': 0.8554955397196855,
        'r': 0.9257905390050531,
        'MSE': 23.62934150612383,
        'RMSE': 4.861002109249062,
        'MAE': 3.2346187227704073
    },
    'CatBoost': {
        'R2': 0.9410264346247154,
        'r': 0.9706774545477034,
        'MSE': 9.643346048856559,
        'RMSE': 3.1053737373875885,
        'MAE': 2.202950907314847
    }
}

# Data normalization
metrics = ['R2', 'r', 'MSE', 'RMSE', 'MAE']

# Calculate min and max for each metric
values = {metric: [data[model][metric] for model in data] for metric in metrics}
min_max = {}

for metric in metrics:
    min_val = min(values[metric])
    max_val = max(values[metric])
    min_max[metric] = (min_val, max_val)

# Normalization function
def normalize(value, metric):
    min_val, max_val = min_max[metric]
    if metric in ['MSE', 'RMSE', 'MAE']:  # Lower is better
        return 1 - ((value - min_val) / (max_val - min_val) if max_val != min_val else 0)
    else:  # Higher is better (R2 and r)
        return (value - min_val) / (max_val - min_val) if max_val != min_val else 1

# Normalized data
normalized_data = {}
for model in data:
    normalized_data[model] = [normalize(data[model][metric], metric) for metric in metrics]

# Radar chart setup
categories = ['R²', 'r', 'MSE', 'RMSE', 'MAE']
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # Close the loop

# Create the plot
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw=dict(polar=True))

# Colors for each model
colors = ['#1f77b4', '#2ca02c', '#d62728', '#ff7f0e', '#9467bd', '#8c564b']

# Plot each model
for i, (model, values_norm) in enumerate(normalized_data.items()):
    values_norm += values_norm[:1]  # Close the loop
    ax.plot(angles, values_norm, 'o-', linewidth=2.5, label=model, color=colors[i])
    ax.fill(angles, values_norm, alpha=0.1, color=colors[i])

# Set category labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=14, fontweight='bold')

# Set y-axis limits and labels
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=11)
ax.set_rlabel_position(30)

# Add grid
ax.grid(True, alpha=0.3)

# Title and legend
plt.title('Model Performance Comparison - Radar Chart\n(Normalized Values: Higher is Better)', 
          fontsize=16, fontweight='bold', pad=20)

# Place legend outside the plot
plt.legend(loc='upper left', bbox_to_anchor=(1.1, 1.0), fontsize=12, framealpha=0.9)

# Add note about normalization
plt.figtext(0.02, 0.02, 
            'Note: MSE, RMSE, MAE are inverted (lower values become higher)\nR² and r are normalized linearly',
            fontsize=10, style='italic', bbox=dict(facecolor='lightgray', alpha=0.5))

# Display the plot
plt.tight_layout()
plt.show()

# Print original values for reference
print("="*60)
print("ORIGINAL VALUES".center(60))
print("="*60)

for model in data:
    print(f"\n{model}:")
    print("-" * 40)
    for metric in metrics:
        print(f"  {metric:6}: {data[model][metric]:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# Model data
data = {
    'AdaBoost': {
        'R2': 0.8936338904609085,
        'r': 0.9454283955070929,
        'MSE': 17.392965740306362,
        'RMSE': 4.170487470345207,
        'MAE': 2.8773415133882962
    },
    'RF': {
        'R2': 0.9010254418819429,
        'r': 0.9499213993386695,
        'MSE': 16.184300675927787,
        'RMSE': 4.02297162256059,
        'MAE': 2.7548876303562193
    },
    'SVR': {
        'R2': 0.795413370570063,
        'r': 0.893487739574029,
        'MSE': 33.453966230586616,
        'RMSE': 5.783940372322887,
        'MAE': 3.2707225028475957
    },
    'XGBoost': {
        'R2': 0.9347851857272262,
        'r': 0.9670153891913157,
        'MSE': 10.663913866191805,
        'RMSE': 3.265564861734001,
        'MAE': 2.409547672208613
    },
    'ANN': {
        'R2': 0.8554955397196855,
        'r': 0.9257905390050531,
        'MSE': 23.62934150612383,
        'RMSE': 4.861002109249062,
        'MAE': 3.2346187227704073
    },
    'CatBoost': {
        'R2': 0.9410264346247154,
        'r': 0.9706774545477034,
        'MSE': 9.643346048856559,
        'RMSE': 3.1053737373875885,
        'MAE': 2.202950907314847
    }
}

# Data normalization
metrics = ['R2', 'r', 'MSE', 'RMSE', 'MAE']

# Calculate min and max for each metric
values = {metric: [data[model][metric] for model in data] for metric in metrics}
min_max = {}

for metric in metrics:
    min_val = min(values[metric])
    max_val = max(values[metric])
    min_max[metric] = (min_val, max_val)

# Normalization function
def normalize(value, metric):
    min_val, max_val = min_max[metric]
    if metric in ['MSE', 'RMSE', 'MAE']:  # Lower is better
        return 1 - ((value - min_val) / (max_val - min_val) if max_val != min_val else 0)
    else:  # Higher is better (R2 and r)
        return (value - min_val) / (max_val - min_val) if max_val != min_val else 1

# Normalized data
normalized_data = {}
for model in data:
    normalized_data[model] = [normalize(data[model][metric], metric) for metric in metrics]

# Radar chart setup
categories = ['R²', 'r', 'MSE', 'RMSE', 'MAE']
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # Close the loop

# Create the plot
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw=dict(polar=True))

# Colors for each model
colors = ['#1f77b4', '#2ca02c', '#d62728', '#ff7f0e', '#9467bd', '#8c564b']

# Plot each model
for i, (model, values_norm) in enumerate(normalized_data.items()):
    values_norm += values_norm[:1]  # Close the loop
    ax.plot(angles, values_norm, 'o-', linewidth=2.5, label=model, color=colors[i])
    ax.fill(angles, values_norm, alpha=0.1, color=colors[i])
    
    # Add small value labels at each point (optional)
    for j, (angle, val) in enumerate(zip(angles[:-1], values_norm[:-1])):
        if val > 0.05:  # Only show if value is significant
            ax.text(angle, val+0.03, f'{data[model][metrics[j]]:.3f}', 
                   fontsize=8, ha='center', va='center', color=colors[i])

# Set category labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=14, fontweight='bold')

# Set y-axis limits and labels
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=11)
ax.set_rlabel_position(30)

# Add grid
ax.grid(True, alpha=0.3)

# Title and legend
plt.title('Model Performance Comparison - Radar Chart\n(Normalized Values: Higher is Better)', 
          fontsize=16, fontweight='bold', pad=20)

# Place legend outside the plot
plt.legend(loc='upper left', bbox_to_anchor=(1.1, 1.0), fontsize=12, framealpha=0.9)

# Add note about normalization
plt.figtext(0.02, 0.02, 
            'Note: MSE, RMSE, MAE are inverted (lower values become higher)\nR² and r are normalized linearly',
            fontsize=10, style='italic', bbox=dict(facecolor='lightgray', alpha=0.5))

# Display the plot
plt.tight_layout()
plt.show()

# Print original values for reference
print("="*60)
print("ORIGINAL VALUES".center(60))
print("="*60)

for model in data:
    print(f"\n{model}:")
    print("-" * 40)
    for metric in metrics:
        print(f"  {metric:6}: {data[model][metric]:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# Model data with specified order
data = {
    'AdaBoost': {
        'R2': 0.8936338904609085,
        'r': 0.9454283955070929,
        'MSE': 17.392965740306362,
        'RMSE': 4.170487470345207,
        'MAE': 2.8773415133882962
    },
    'RF': {
        'R2': 0.9010254418819429,
        'r': 0.9499213993386695,
        'MSE': 16.184300675927787,
        'RMSE': 4.02297162256059,
        'MAE': 2.7548876303562193
    },
    'SVR': {
        'R2': 0.795413370570063,
        'r': 0.893487739574029,
        'MSE': 33.453966230586616,
        'RMSE': 5.783940372322887,
        'MAE': 3.2707225028475957
    },
    'XGBoost': {
        'R2': 0.9347851857272262,
        'r': 0.9670153891913157,
        'MSE': 10.663913866191805,
        'RMSE': 3.265564861734001,
        'MAE': 2.409547672208613
    },
    'ANN': {
        'R2': 0.8554955397196855,
        'r': 0.9257905390050531,
        'MSE': 23.62934150612383,
        'RMSE': 4.861002109249062,
        'MAE': 3.2346187227704073
    },
    'CatBoost': {
        'R2': 0.9410264346247154,
        'r': 0.9706774545477034,
        'MSE': 9.643346048856559,
        'RMSE': 3.1053737373875885,
        'MAE': 2.202950907314847
    }
}

# Specify the exact order of models
model_order = ['AdaBoost', 'RF', 'SVR', 'XGBoost', 'ANN', 'CatBoost']

# Reorder data according to specified order
ordered_data = {model: data[model] for model in model_order}

# Data normalization
metrics = ['R2', 'r', 'MSE', 'RMSE', 'MAE']

# Calculate min and max for each metric
values = {metric: [data[model][metric] for model in data] for metric in metrics}
min_max = {}

for metric in metrics:
    min_val = min(values[metric])
    max_val = max(values[metric])
    min_max[metric] = (min_val, max_val)

# Normalization function
def normalize(value, metric):
    min_val, max_val = min_max[metric]
    if metric in ['MSE', 'RMSE', 'MAE']:  # Lower is better
        return 1 - ((value - min_val) / (max_val - min_val) if max_val != min_val else 0)
    else:  # Higher is better (R2 and r)
        return (value - min_val) / (max_val - min_val) if max_val != min_val else 1

# Normalized data
normalized_data = {}
for model in ordered_data:
    normalized_data[model] = [normalize(ordered_data[model][metric], metric) for metric in metrics]

# Radar chart setup
categories = ['R²', 'r', 'MSE', 'RMSE', 'MAE']
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # Close the loop

# Create the plot
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw=dict(polar=True))

# Colors for each model
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

# Plot each model in the specified order
for i, (model, values_norm) in enumerate(normalized_data.items()):
    values_norm += values_norm[:1]  # Close the loop
    ax.plot(angles, values_norm, 'o-', linewidth=2.5, label=model, color=colors[i])
    ax.fill(angles, values_norm, alpha=0.1, color=colors[i])

# Set category labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=14, fontweight='bold')

# Set y-axis limits and labels
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=11)
ax.set_rlabel_position(30)

# Add grid
ax.grid(True, alpha=0.3)

# Title and legend
plt.title('Model Performance Comparison - Radar Chart\n(Normalized Values: Higher is Better)', 
          fontsize=16, fontweight='bold', pad=20)

# Place legend outside the plot
plt.legend(loc='upper left', bbox_to_anchor=(1.1, 1.0), fontsize=12, framealpha=0.9)

# Add note about normalization
plt.figtext(0.02, 0.02, 
            'Note: MSE, RMSE, MAE are inverted (lower values become higher)\nR² and r are normalized linearly',
            fontsize=10, style='italic', bbox=dict(facecolor='lightgray', alpha=0.5))

# Display the plot
plt.tight_layout()
plt.show()

# Print original values in the specified order
print("="*70)
print("MODEL PERFORMANCE METRICS".center(70))
print("="*70)

for model in model_order:
    print(f"\n{model}:")
    print("-" * 50)
    for metric in metrics:
        print(f"  {metric:6}: {data[model][metric]:.4f}")

# Find best and worst models
print("\n" + "="*70)
print("SUMMARY".center(70))
print("="*70)

# Calculate average normalized score for each model
avg_scores = {}
for model in model_order:
    avg_scores[model] = sum(normalized_data[model]) / len(normalized_data[model])

best_model = max(avg_scores, key=avg_scores.get)
worst_model = min(avg_scores, key=avg_scores.get)

print(f"\nBest Overall Model: {best_model} (Score: {avg_scores[best_model]:.3f})")
print(f"Worst Overall Model: {worst_model} (Score: {avg_scores[worst_model]:.3f})")

print("\n" + "="*70)
print("RANKING (Best to Worst)".center(70))
print("="*70)

# Sort models by average score
ranked_models = sorted(avg_scores.items(), key=lambda x: x[1], reverse=True)
for i, (model, score) in enumerate(ranked_models, 1):
    print(f"{i}. {model}: {score:.3f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# Model data
data = {
    'AdaBoost': {
        'R2': 0.8936338904609085,
        'r': 0.9454283955070929,
        'MSE': 17.392965740306362,
        'RMSE': 4.170487470345207,
        'MAE': 2.8773415133882962
    },
    'RF': {
        'R2': 0.9010254418819429,
        'r': 0.9499213993386695,
        'MSE': 16.184300675927787,
        'RMSE': 4.02297162256059,
        'MAE': 2.7548876303562193
    },
    'SVR': {
        'R2': 0.795413370570063,
        'r': 0.893487739574029,
        'MSE': 33.453966230586616,
        'RMSE': 5.783940372322887,
        'MAE': 3.2707225028475957
    },
    'XGBoost': {
        'R2': 0.9347851857272262,
        'r': 0.9670153891913157,
        'MSE': 10.663913866191805,
        'RMSE': 3.265564861734001,
        'MAE': 2.409547672208613
    },
    'ANN': {
        'R2': 0.8554955397196855,
        'r': 0.9257905390050531,
        'MSE': 23.62934150612383,
        'RMSE': 4.861002109249062,
        'MAE': 3.2346187227704073
    },
    'CatBoost': {
        'R2': 0.9410264346247154,
        'r': 0.9706774545477034,
        'MSE': 9.643346048856559,
        'RMSE': 3.1053737373875885,
        'MAE': 2.202950907314847
    }
}

# Fixed model order and colors
model_order = ['AdaBoost', 'RF', 'SVR', 'XGBoost', 'ANN', 'CatBoost']
model_colors = {
    'AdaBoost': '#1f77b4',  # blue
    'RF': '#ff7f0e',        # orange
    'SVR': '#2ca02c',       # green
    'XGBoost': '#d62728',   # red
    'ANN': '#9467bd',       # purple
    'CatBoost': '#8c564b'   # brown
}

# Data normalization
metrics = ['R2', 'r', 'MSE', 'RMSE', 'MAE']

# Calculate min and max for each metric
values = {metric: [data[model][metric] for model in data] for metric in metrics}
min_max = {}

for metric in metrics:
    min_val = min(values[metric])
    max_val = max(values[metric])
    min_max[metric] = (min_val, max_val)

# Normalization function
def normalize(value, metric):
    min_val, max_val = min_max[metric]
    if metric in ['MSE', 'RMSE', 'MAE']:  # Lower is better
        return 1 - ((value - min_val) / (max_val - min_val) if max_val != min_val else 0)
    else:  # Higher is better (R2 and r)
        return (value - min_val) / (max_val - min_val) if max_val != min_val else 1

# Normalized data
normalized_data = {}
for model in data:
    normalized_data[model] = [normalize(data[model][metric], metric) for metric in metrics]

# Radar chart setup
categories = ['R²', 'r', 'MSE', 'RMSE', 'MAE']
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # Close the loop

# Create the plot
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw=dict(polar=True))

# Plot each model in the fixed order with fixed colors
for model in model_order:
    values_norm = normalized_data[model] + normalized_data[model][:1]  # Close the loop
    ax.plot(angles, values_norm, 'o-', linewidth=2.5, label=model, color=model_colors[model])
    ax.fill(angles, values_norm, alpha=0.1, color=model_colors[model])

# Set category labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=14, fontweight='bold')

# Set y-axis limits and labels
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=11)
ax.set_rlabel_position(30)

# Add grid
ax.grid(True, alpha=0.3)

# Title and legend
plt.title('Model Performance Comparison - Radar Chart\n(Normalized Values: Higher is Better)', 
          fontsize=16, fontweight='bold', pad=20)

# Place legend outside the plot with fixed order
handles = [plt.Line2D([0], [0], color=model_colors[model], lw=2.5) for model in model_order]
plt.legend(handles, model_order, loc='upper left', bbox_to_anchor=(1.1, 1.0), 
          fontsize=12, framealpha=0.9)

# Add note about normalization
plt.figtext(0.02, 0.02, 
            'Note: MSE, RMSE, MAE are inverted (lower values become higher)\nR² and r are normalized linearly',
            fontsize=10, style='italic', bbox=dict(facecolor='lightgray', alpha=0.5))

# Display the plot
plt.tight_layout()
plt.show()

# Print original values in the fixed order
print("="*70)
print("MODEL PERFORMANCE METRICS".center(70))
print("="*70)

for model in model_order:
    print(f"\n{model}:")
    print("-" * 50)
    for metric in metrics:
        print(f"  {metric:6}: {data[model][metric]:.4f}")

# Find best and worst models
print("\n" + "="*70)
print("SUMMARY".center(70))
print("="*70)

# Calculate average normalized score for each model
avg_scores = {}
for model in model_order:
    avg_scores[model] = sum(normalized_data[model]) / len(normalized_data[model])

best_model = max(avg_scores, key=avg_scores.get)
worst_model = min(avg_scores, key=avg_scores.get)

print(f"\nBest Overall Model: {best_model} (Score: {avg_scores[best_model]:.3f})")
print(f"Worst Overall Model: {worst_model} (Score: {avg_scores[worst_model]:.3f})")

print("\n" + "="*70)
print("RANKING (Best to Worst)".center(70))
print("="*70)

# Sort models by average score
ranked_models = sorted(avg_scores.items(), key=lambda x: x[1], reverse=True)
for i, (model, score) in enumerate(ranked_models, 1):
    print(f"{i}. {model}: {score:.3f}")

In [ ]:
# hpc
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# HPC Model data with specified order
data = {
    'AdaBoost': {
        'R2': 0.8712785408848976,
        'r': 0.934919665428412,
        'MSE': 34.8669612619576,
        'RMSE': 5.904825252448847,
        'MAE': 4.018362875336771
    },
    'RF': {
        'R2': 0.8978752044365991,
        'r': 0.9514188308631276,
        'MSE': 27.662685889929183,
        'RMSE': 5.259532858527379,
        'MAE': 3.5502451501972643
    },
    'SVR': {
        'R2': 0.826605492069586,
        'r': 0.9094449306547474,
        'MSE': 46.96761233602753,
        'RMSE': 6.853292080163191,
        'MAE': 4.970869310220807
    },
    'XGBoost': {
        'R2': 0.9430101920623648,
        'r': 0.971704893291186,
        'MSE': 15.436908805633607,
        'RMSE': 3.928983177061669,
        'MAE': 2.6047408697217005
    },
    'ANN': {
        'R2': 0.9036603020573442,
        'r': 0.9513303365648808,
        'MSE': 26.095668424264854,
        'RMSE': 5.108391960711791,
        'MAE': 3.474560702049442
    },
    'CatBoost': {
        'R2': 0.9392762646232117,
        'r': 0.9697326539322698,
        'MSE': 16.448322941791726,
        'RMSE': 4.055653207781914,
        'MAE': 2.713028433306617
    }
}

# Fixed model order and colors (same as previous)
model_order = ['AdaBoost', 'RF', 'SVR', 'XGBoost', 'ANN', 'CatBoost']
model_colors = {
    'AdaBoost': '#1f77b4',  # blue
    'RF': '#ff7f0e',        # orange
    'SVR': '#2ca02c',       # green
    'XGBoost': '#d62728',   # red
    'ANN': '#9467bd',       # purple
    'CatBoost': '#8c564b'   # brown
}

# Data normalization
metrics = ['R2', 'r', 'MSE', 'RMSE', 'MAE']

# Calculate min and max for each metric
values = {metric: [data[model][metric] for model in data] for metric in metrics}
min_max = {}

for metric in metrics:
    min_val = min(values[metric])
    max_val = max(values[metric])
    min_max[metric] = (min_val, max_val)

# Print min-max values for reference
print("NORMALIZATION RANGES:")
for metric in metrics:
    print(f"  {metric}: min={min_max[metric][0]:.4f}, max={min_max[metric][1]:.4f}")

# Normalization function
def normalize(value, metric):
    min_val, max_val = min_max[metric]
    if metric in ['MSE', 'RMSE', 'MAE']:  # Lower is better
        return 1 - ((value - min_val) / (max_val - min_val) if max_val != min_val else 0)
    else:  # Higher is better (R2 and r)
        return (value - min_val) / (max_val - min_val) if max_val != min_val else 1

# Normalized data
normalized_data = {}
for model in data:
    normalized_data[model] = [normalize(data[model][metric], metric) for metric in metrics]

# Radar chart setup
categories = ['R²', 'r', 'MSE', 'RMSE', 'MAE']
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # Close the loop

# Create the plot
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw=dict(polar=True))

# Plot each model in the fixed order with fixed colors
for model in model_order:
    values_norm = normalized_data[model] + normalized_data[model][:1]  # Close the loop
    ax.plot(angles, values_norm, 'o-', linewidth=2.5, label=model, color=model_colors[model])
    ax.fill(angles, values_norm, alpha=0.1, color=model_colors[model])

# Set category labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=14, fontweight='bold')

# Set y-axis limits and labels
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=11)
ax.set_rlabel_position(30)

# Add grid
ax.grid(True, alpha=0.3)

# Title and legend
plt.title('HPC Models Performance Comparison - Radar Chart\n(Normalized Values: Higher is Better)', 
          fontsize=16, fontweight='bold', pad=20)

# Place legend outside the plot with fixed order
handles = [plt.Line2D([0], [0], color=model_colors[model], lw=2.5) for model in model_order]
plt.legend(handles, model_order, loc='upper left', bbox_to_anchor=(1.1, 1.0), 
          fontsize=12, framealpha=0.9)

# Add note about normalization
plt.figtext(0.02, 0.02, 
            'Note: MSE, RMSE, MAE are inverted (lower values become higher)\nR² and r are normalized linearly',
            fontsize=10, style='italic', bbox=dict(facecolor='lightgray', alpha=0.5))

# Display the plot
plt.tight_layout()
plt.show()

# Print original values in the fixed order
print("\n" + "="*70)
print("HPC MODELS - ORIGINAL VALUES".center(70))
print("="*70)

for model in model_order:
    print(f"\n{model}:")
    print("-" * 50)
    for metric in metrics:
        print(f"  {metric:6}: {data[model][metric]:.4f}")

# Find best and worst models
print("\n" + "="*70)
print("SUMMARY".center(70))
print("="*70)

# Calculate average normalized score for each model
avg_scores = {}
for model in model_order:
    avg_scores[model] = sum(normalized_data[model]) / len(normalized_data[model])

best_model = max(avg_scores, key=avg_scores.get)
worst_model = min(avg_scores, key=avg_scores.get)

print(f"\nBest Overall Model: {best_model} (Score: {avg_scores[best_model]:.3f})")
print(f"Worst Overall Model: {worst_model} (Score: {avg_scores[worst_model]:.3f})")

print("\n" + "="*70)
print("RANKING (Best to Worst)".center(70))
print("="*70)

# Sort models by average score
ranked_models = sorted(avg_scores.items(), key=lambda x: x[1], reverse=True)
for i, (model, score) in enumerate(ranked_models, 1):
    print(f"{i}. {model}: {score:.3f}")

# Print normalized values for reference
print("\n" + "="*70)
print("NORMALIZED VALUES (1 = Best Performance)".center(70))
print("="*70)

for model in model_order:
    print(f"\n{model}:")
    print("-" * 50)
    for i, metric in enumerate(metrics):
        print(f"  {metric:6}: {normalized_data[model][i]:.4f}")

In [ ]:
#hsc
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# HSC Model data with specified order
data = {
    'AdaBoost': {
        'R2': 0.940217509003306,
        'r': 0.970559829196228,
        'MSE': 18.58718750000001,
        'RMSE': 4.3112860610263395,
        'MAE': 2.4786111111111113
    },
    'RF': {
        'R2': 0.9178374597049438,
        'r': 0.9627543074652203,
        'MSE': 25.545448449529648,
        'RMSE': 5.054250532920746,
        'MAE': 3.065495495495474
    },
    'SVR': {
        'R2': 0.8700653555824004,
        'r': 0.9389573519384933,
        'MSE': 40.39844372944098,
        'RMSE': 6.355977008253017,
        'MAE': 4.631120156371181
    },
    'XGBoost': {
        'R2': 0.9481105964399211,
        'r': 0.9740561112281306,
        'MSE': 16.13311953307015,
        'RMSE': 4.016605473913283,
        'MAE': 2.743214022318521
    },
    'ANN': {
        'R2': 0.9304629135987174,
        'r': 0.9678659469952312,
        'MSE': 21.62002354863096,
        'RMSE': 4.649733707281629,
        'MAE': 2.996058530518593
    },
    'CatBoost': {
        'R2': 0.9479394521576345,
        'r': 0.9739805121615106,
        'MSE': 16.186330612290494,
        'RMSE': 4.023223907799626,
        'MAE': 2.701707870474475
    }
}

# Fixed model order and colors (same as previous)
model_order = ['AdaBoost', 'RF', 'SVR', 'XGBoost', 'ANN', 'CatBoost']
model_colors = {
    'AdaBoost': '#1f77b4',  # blue
    'RF': '#ff7f0e',        # orange
    'SVR': '#2ca02c',       # green
    'XGBoost': '#d62728',   # red
    'ANN': '#9467bd',       # purple
    'CatBoost': '#8c564b'   # brown
}

# Data normalization
metrics = ['R2', 'r', 'MSE', 'RMSE', 'MAE']

# Calculate min and max for each metric
values = {metric: [data[model][metric] for model in data] for metric in metrics}
min_max = {}

for metric in metrics:
    min_val = min(values[metric])
    max_val = max(values[metric])
    min_max[metric] = (min_val, max_val)

# Print min-max values for reference
print("HSC DATA - NORMALIZATION RANGES:")
print("="*50)
for metric in metrics:
    print(f"  {metric}: min={min_max[metric][0]:.4f}, max={min_max[metric][1]:.4f}")

# Normalization function
def normalize(value, metric):
    min_val, max_val = min_max[metric]
    if metric in ['MSE', 'RMSE', 'MAE']:  # Lower is better
        return 1 - ((value - min_val) / (max_val - min_val) if max_val != min_val else 0)
    else:  # Higher is better (R2 and r)
        return (value - min_val) / (max_val - min_val) if max_val != min_val else 1

# Normalized data
normalized_data = {}
for model in data:
    normalized_data[model] = [normalize(data[model][metric], metric) for metric in metrics]

# Radar chart setup
categories = ['R²', 'r', 'MSE', 'RMSE', 'MAE']
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # Close the loop

# Create the plot
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw=dict(polar=True))

# Plot each model in the fixed order with fixed colors
for model in model_order:
    values_norm = normalized_data[model] + normalized_data[model][:1]  # Close the loop
    ax.plot(angles, values_norm, 'o-', linewidth=2.5, label=model, color=model_colors[model])
    ax.fill(angles, values_norm, alpha=0.1, color=model_colors[model])

# Set category labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=14, fontweight='bold')

# Set y-axis limits and labels
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=11)
ax.set_rlabel_position(30)

# Add grid
ax.grid(True, alpha=0.3)

# Title and legend
plt.title('HSC Models Performance Comparison - Radar Chart\n(Normalized Values: Higher is Better)', 
          fontsize=16, fontweight='bold', pad=20)

# Place legend outside the plot with fixed order
handles = [plt.Line2D([0], [0], color=model_colors[model], lw=2.5) for model in model_order]
plt.legend(handles, model_order, loc='upper left', bbox_to_anchor=(1.1, 1.0), 
          fontsize=12, framealpha=0.9)

# Add note about normalization
plt.figtext(0.02, 0.02, 
            'Note: MSE, RMSE, MAE are inverted (lower values become higher)\nR² and r are normalized linearly',
            fontsize=10, style='italic', bbox=dict(facecolor='lightgray', alpha=0.5))

# Display the plot
plt.tight_layout()
plt.show()

# Print original values in the fixed order
print("\n" + "="*70)
print("HSC MODELS - ORIGINAL VALUES".center(70))
print("="*70)

for model in model_order:
    print(f"\n{model}:")
    print("-" * 50)
    for metric in metrics:
        print(f"  {metric:6}: {data[model][metric]:.4f}")

# Find best and worst models
print("\n" + "="*70)
print("SUMMARY".center(70))
print("="*70)

# Calculate average normalized score for each model
avg_scores = {}
for model in model_order:
    avg_scores[model] = sum(normalized_data[model]) / len(normalized_data[model])

best_model = max(avg_scores, key=avg_scores.get)
worst_model = min(avg_scores, key=avg_scores.get)

print(f"\nBest Overall Model: {best_model} (Score: {avg_scores[best_model]:.3f})")
print(f"Worst Overall Model: {worst_model} (Score: {avg_scores[worst_model]:.3f})")

print("\n" + "="*70)
print("RANKING (Best to Worst)".center(70))
print("="*70)

# Sort models by average score
ranked_models = sorted(avg_scores.items(), key=lambda x: x[1], reverse=True)
for i, (model, score) in enumerate(ranked_models, 1):
    print(f"{i}. {model}: {score:.3f}")

# Print normalized values for reference
print("\n" + "="*70)
print("NORMALIZED VALUES (1 = Best Performance)".center(70))
print("="*70)

for model in model_order:
    print(f"\n{model}:")
    print("-" * 50)
    for i, metric in enumerate(metrics):
        print(f"  {metric:6}: {normalized_data[model][i]:.4f}")

In [ ]:
#uhpc
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# UHPC Model data with specified order
data = {
    'AdaBoost': {
        'R2': 0.949530631342127,
        'r': 0.9754925634337887,
        'MSE': 78.30537238916332,
        'RMSE': 8.849032285462819,
        'MAE': 6.82382332532984
    },
    'RF': {
        'R2': 0.9481868404666568,
        'r': 0.9742978244949422,
        'MSE': 80.39032109597541,
        'RMSE': 8.96606497277236,
        'MAE': 6.48810513006559
    },
    'SVR': {
        'R2': 0.6797961784410069,
        'r': 0.8257191736936643,
        'MSE': 496.809850299143,
        'RMSE': 22.28923171172894,
        'MAE': 17.51849262425527
    },
    'XGBoost': {
        'R2': 0.9639396762516196,
        'r': 0.9821105668558147,
        'MSE': 55.94912626572412,
        'RMSE': 7.479914856849917,
        'MAE': 4.865019280242921
    },
    'ANN': {
        'R2': 0.9397615127958833,
        'r': 0.9695563158484903,
        'MSE': 93.46257538219425,
        'RMSE': 9.66760442830561,
        'MAE': 6.876883967105487
    },
    'CatBoost': {
        'R2': 0.9731531628560162,
        'r': 0.9869335083985321,
        'MSE': 41.65400986649618,
        'RMSE': 6.453991777690469,
        'MAE': 4.385934637727605
    }
}

# Fixed model order and colors (same as previous)
model_order = ['AdaBoost', 'RF', 'SVR', 'XGBoost', 'ANN', 'CatBoost']
model_colors = {
    'AdaBoost': '#1f77b4',  # blue
    'RF': '#ff7f0e',        # orange
    'SVR': '#2ca02c',       # green
    'XGBoost': '#d62728',   # red
    'ANN': '#9467bd',       # purple
    'CatBoost': '#8c564b'   # brown
}

# Data normalization
metrics = ['R2', 'r', 'MSE', 'RMSE', 'MAE']

# Calculate min and max for each metric
values = {metric: [data[model][metric] for model in data] for metric in metrics}
min_max = {}

for metric in metrics:
    min_val = min(values[metric])
    max_val = max(values[metric])
    min_max[metric] = (min_val, max_val)

# Print min-max values for reference
print("UHPC DATA - NORMALIZATION RANGES:")
print("="*60)
for metric in metrics:
    print(f"  {metric}: min={min_max[metric][0]:.4f}, max={min_max[metric][1]:.4f}")

# Normalization function
def normalize(value, metric):
    min_val, max_val = min_max[metric]
    if metric in ['MSE', 'RMSE', 'MAE']:  # Lower is better
        return 1 - ((value - min_val) / (max_val - min_val) if max_val != min_val else 0)
    else:  # Higher is better (R2 and r)
        return (value - min_val) / (max_val - min_val) if max_val != min_val else 1

# Normalized data
normalized_data = {}
for model in data:
    normalized_data[model] = [normalize(data[model][metric], metric) for metric in metrics]

# Radar chart setup
categories = ['R²', 'r', 'MSE', 'RMSE', 'MAE']
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # Close the loop

# Create the plot
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw=dict(polar=True))

# Plot each model in the fixed order with fixed colors
for model in model_order:
    values_norm = normalized_data[model] + normalized_data[model][:1]  # Close the loop
    ax.plot(angles, values_norm, 'o-', linewidth=2.5, label=model, color=model_colors[model])
    ax.fill(angles, values_norm, alpha=0.1, color=model_colors[model])

# Set category labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=14, fontweight='bold')

# Set y-axis limits and labels
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=11)
ax.set_rlabel_position(30)

# Add grid
ax.grid(True, alpha=0.3)

# Title and legend
plt.title('UHPC Models Performance Comparison - Radar Chart\n(Normalized Values: Higher is Better)', 
          fontsize=16, fontweight='bold', pad=20)

# Place legend outside the plot with fixed order
handles = [plt.Line2D([0], [0], color=model_colors[model], lw=2.5) for model in model_order]
plt.legend(handles, model_order, loc='upper left', bbox_to_anchor=(1.1, 1.0), 
          fontsize=12, framealpha=0.9)

# Add note about normalization
plt.figtext(0.02, 0.02, 
            'Note: MSE, RMSE, MAE are inverted (lower values become higher)\nR² and r are normalized linearly',
            fontsize=10, style='italic', bbox=dict(facecolor='lightgray', alpha=0.5))

# Display the plot
plt.tight_layout()
plt.show()

# Print original values in the fixed order
print("\n" + "="*70)
print("UHPC MODELS - ORIGINAL VALUES".center(70))
print("="*70)

for model in model_order:
    print(f"\n{model}:")
    print("-" * 50)
    for metric in metrics:
        print(f"  {metric:6}: {data[model][metric]:.4f}")

# Find best and worst models
print("\n" + "="*70)
print("SUMMARY".center(70))
print("="*70)

# Calculate average normalized score for each model
avg_scores = {}
for model in model_order:
    avg_scores[model] = sum(normalized_data[model]) / len(normalized_data[model])

best_model = max(avg_scores, key=avg_scores.get)
worst_model = min(avg_scores, key=avg_scores.get)

print(f"\nBest Overall Model: {best_model} (Score: {avg_scores[best_model]:.3f})")
print(f"Worst Overall Model: {worst_model} (Score: {avg_scores[worst_model]:.3f})")

print("\n" + "="*70)
print("RANKING (Best to Worst)".center(70))
print("="*70)

# Sort models by average score
ranked_models = sorted(avg_scores.items(), key=lambda x: x[1], reverse=True)
for i, (model, score) in enumerate(ranked_models, 1):
    print(f"{i}. {model}: {score:.3f}")

# Print normalized values for reference
print("\n" + "="*70)
print("NORMALIZED VALUES (1 = Best Performance)".center(70))
print("="*70)

for model in model_order:
    print(f"\n{model}:")
    print("-" * 50)
    for i, metric in enumerate(metrics):
        print(f"  {metric:6}: {normalized_data[model][i]:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# C188 Model data with specified order
data = {
    'AdaBoost': {
        'R2': 0.8864089019950795,
        'r': 0.9426060170791023,
        'MSE': 36.27360643199691,
        'RMSE': 6.022757377812667,
        'MAE': 4.48366362715299
    },
    'RF': {
        'R2': 0.9026244917908295,
        'r': 0.9620749292743389,
        'MSE': 31.095402042351346,
        'RMSE': 5.57632513779024,
        'MAE': 3.934495682855961
    },
    'SVR': {
        'R2': 0.9128014509267974,
        'r': 0.9641634751087838,
        'MSE': 27.845543410325206,
        'RMSE': 5.276887663227749,
        'MAE': 3.836639427334917
    },
    'XGBoost': {
        'R2': 0.9275176082607819,
        'r': 0.9672814220719048,
        'MSE': 23.14616019544357,
        'RMSE': 4.811045644705896,
        'MAE': 3.7146837258846204
    },
    'ANN': {
        'R2': 0.8390106305083072,
        'r': 0.9242232667162509,
        'MSE': 51.40953059916753,
        'RMSE': 7.1700439747024935,
        'MAE': 5.791798248594065
    },
    'CatBoost': {
        'R2': 0.9510351924805193,
        'r': 0.9773135222635858,
        'MSE': 15.63617385671532,
        'RMSE': 3.9542602161106344,
        'MAE': 2.8342888902332084
    }
}

# Fixed model order and colors (same as previous)
model_order = ['AdaBoost', 'RF', 'SVR', 'XGBoost', 'ANN', 'CatBoost']
model_colors = {
    'AdaBoost': '#1f77b4',  # blue
    'RF': '#ff7f0e',        # orange
    'SVR': '#2ca02c',       # green
    'XGBoost': '#d62728',   # red
    'ANN': '#9467bd',       # purple
    'CatBoost': '#8c564b'   # brown
}

# Data normalization
metrics = ['R2', 'r', 'MSE', 'RMSE', 'MAE']

# Calculate min and max for each metric
values = {metric: [data[model][metric] for model in data] for metric in metrics}
min_max = {}

for metric in metrics:
    min_val = min(values[metric])
    max_val = max(values[metric])
    min_max[metric] = (min_val, max_val)

# Print min-max values for reference
print("C188 DATA - NORMALIZATION RANGES:")
print("="*60)
for metric in metrics:
    print(f"  {metric}: min={min_max[metric][0]:.4f}, max={min_max[metric][1]:.4f}")

# Normalization function
def normalize(value, metric):
    min_val, max_val = min_max[metric]
    if metric in ['MSE', 'RMSE', 'MAE']:  # Lower is better
        return 1 - ((value - min_val) / (max_val - min_val) if max_val != min_val else 0)
    else:  # Higher is better (R2 and r)
        return (value - min_val) / (max_val - min_val) if max_val != min_val else 1

# Normalized data
normalized_data = {}
for model in data:
    normalized_data[model] = [normalize(data[model][metric], metric) for metric in metrics]

# Radar chart setup
categories = ['R²', 'r', 'MSE', 'RMSE', 'MAE']
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # Close the loop

# Create the plot
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw=dict(polar=True))

# Plot each model in the fixed order with fixed colors
for model in model_order:
    values_norm = normalized_data[model] + normalized_data[model][:1]  # Close the loop
    ax.plot(angles, values_norm, 'o-', linewidth=2.5, label=model, color=model_colors[model])
    ax.fill(angles, values_norm, alpha=0.1, color=model_colors[model])

# Set category labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=14, fontweight='bold')

# Set y-axis limits and labels
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=11)
ax.set_rlabel_position(30)

# Add grid
ax.grid(True, alpha=0.3)

# Title and legend
plt.title('Recycled Aggregate Concrete Models Performance Comparison - Radar Chart\n(Normalized Values: Higher is Better)', 
          fontsize=16, fontweight='bold', pad=20)

# Place legend outside the plot with fixed order
handles = [plt.Line2D([0], [0], color=model_colors[model], lw=2.5) for model in model_order]
plt.legend(handles, model_order, loc='upper left', bbox_to_anchor=(1.1, 1.0), 
          fontsize=12, framealpha=0.9)

# Add note about normalization
plt.figtext(0.02, 0.02, 
            'Note: MSE, RMSE, MAE are inverted (lower values become higher)\nR² and r are normalized linearly',
            fontsize=10, style='italic', bbox=dict(facecolor='lightgray', alpha=0.5))

# Display the plot
plt.tight_layout()
plt.show()

# Print original values in the fixed order
print("\n" + "="*70)
print("C188 MODELS - ORIGINAL VALUES".center(70))
print("="*70)

for model in model_order:
    print(f"\n{model}:")
    print("-" * 50)
    for metric in metrics:
        print(f"  {metric:6}: {data[model][metric]:.4f}")

# Find best and worst models
print("\n" + "="*70)
print("SUMMARY".center(70))
print("="*70)

# Calculate average normalized score for each model
avg_scores = {}
for model in model_order:
    avg_scores[model] = sum(normalized_data[model]) / len(normalized_data[model])

best_model = max(avg_scores, key=avg_scores.get)
worst_model = min(avg_scores, key=avg_scores.get)

print(f"\nBest Overall Model: {best_model} (Score: {avg_scores[best_model]:.3f})")
print(f"Worst Overall Model: {worst_model} (Score: {avg_scores[worst_model]:.3f})")

print("\n" + "="*70)
print("RANKING (Best to Worst)".center(70))
print("="*70)

# Sort models by average score
ranked_models = sorted(avg_scores.items(), key=lambda x: x[1], reverse=True)
for i, (model, score) in enumerate(ranked_models, 1):
    print(f"{i}. {model}: {score:.3f}")

# Print normalized values for reference
print("\n" + "="*70)
print("NORMALIZED VALUES (1 = Best Performance)".center(70))
print("="*70)

for model in model_order:
    print(f"\n{model}:")
    print("-" * 50)
    for i, metric in enumerate(metrics):
        print(f"  {metric:6}: {normalized_data[model][i]:.4f}")